# 01 · Идентификация и re-ID: шесть интерактивных опытов
Лекция 6. Время: 50–60 минут. CPU, без загрузки данных и весов.
Все векторы и scores ниже **синтетические**: они показывают геометрию и ошибки протокола, а не качество FaceNet, ArcFace или OSNet. Демонстрация не принимает фотографии и не распознает людей.

В Jupyter выполните все ячейки. Если управление не появляется, установите `numpy matplotlib ipywidgets` в среду ядра и перезапустите его. На GitHub элементы управления не работают; используйте JupyterLab или загрузите notebook в Colab.

После появления каждого стенда прочитайте вопросы **под ним**. Запишите прогноз **до** изменения параметров, затем измерение и объяснение. В опытах 2, 5 и 6 сравнивайте одни и те же данные при изменении одного фактора.

In [ ]:
# Запускайте первой. В Colab сохраняем штатные PyTorch/CUDA и ipywidgets.
import importlib.util, subprocess, sys
if sys.version_info < (3, 10):
    raise RuntimeError('Нужен Python 3.10 или новее.')
_packages = {'numpy': 'numpy>=1.26,<3', 'matplotlib': 'matplotlib>=3.8,<4', 'ipywidgets': 'ipywidgets>=7.7,<9'}
_missing = [spec for module, spec in _packages.items()
            if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])
    raise RuntimeError('Пакеты установлены. Перезапустите среду выполнения и выполните ноутбук сверху.')
import numpy, matplotlib
if not (1 <= int(numpy.__version__.split('.')[0]) < 3):
    raise RuntimeError('Нужен NumPy 1.x или 2.x; используйте отдельное окружение.')
if int(matplotlib.__version__.split('.')[0]) != 3:
    raise RuntimeError('Нужен Matplotlib 3.x; используйте отдельное окружение.')
_versions = {'python': sys.version.split()[0], 'numpy': numpy.__version__,
             'matplotlib': matplotlib.__version__}
if 'scipy' in _packages:
    import scipy
    if int(scipy.__version__.split('.')[0]) != 1:
        raise RuntimeError('Нужен SciPy 1.x; используйте отдельное окружение.')
    _versions['scipy'] = scipy.__version__
if 'cv2' in _packages:
    import cv2
    _versions['opencv'] = cv2.__version__
    _required_features = ()
    _absent = [name for name in _required_features if not hasattr(cv2, name)]
    if _absent:
        raise RuntimeError('В OpenCV нет ' + ', '.join(_absent) +
                           '. Установите один вариант OpenCV 4.12.0.88 в отдельной среде и перезапустите ядро.')
if 'ipywidgets' in _packages:
    import ipywidgets
    _versions['ipywidgets'] = ipywidgets.__version__
    if int(ipywidgets.__version__.split('.')[0]) not in (7, 8):
        raise RuntimeError('Нужен ipywidgets 7.x или 8.x; в Colab 2026.07 оставьте штатный 7.7.1.')
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except ImportError:
        pass
print('Окружение:', _versions)
print('Вычисления этой лабораторной: CPU; GPU не требуется.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import time, json, csv, hashlib
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({'figure.dpi':110, 'font.size':11})

In [ ]:
import ipywidgets as W
from IPython.display import display
from IPython.utils.capture import capture_output
PANELS={}
def panel(name, fn, controls):
    out=W.Output(); state={'calls':0,'last':None}
    def refresh(change=None):
        with capture_output(display=True) as captured:
            result=fn(**{key:control.value for key,control in controls.items()})
        items=[{'output_type':'display_data','data':item.data,'metadata':item.metadata}
               for item in captured.outputs]
        if captured.stdout:
            items.append({'output_type':'stream','name':'stdout','text':captured.stdout})
        if captured.stderr:
            items.append({'output_type':'stream','name':'stderr','text':captured.stderr})
        out.outputs=tuple(items)
        state['last']=result; state['calls']+=1
    for control in controls.values():
        if hasattr(control,'continuous_update'): control.continuous_update=True
        control.observe(refresh,names='value')
    button=W.Button(description='Повторить опыт',icon='refresh')
    button.on_click(refresh)
    PANELS[name]={'controls':controls,'state':state,'refresh':refresh,'button':button}
    display(W.VBox([W.HBox(list(controls.values()),
                           layout=W.Layout(flex_flow='row wrap')),button,out]))
    refresh()

## 1. Почему нормировка важна
Для ненулевых векторов $s(a,b)=a^Tb/(\|a\|\|b\|)$; после L2-нормировки $\|a-b\|^2=2-2s$. Dot product растет с длиной даже при том же направлении. Нулевой вектор не задает направление.

In [ ]:
def show_cosine(angle, length):
    a=np.array([1.,0.]); b=length*np.array([np.cos(np.deg2rad(angle)),np.sin(np.deg2rad(angle))])
    dot=float(a@b); cosine=None if length==0 else float(dot/np.linalg.norm(b))
    fig,ax=plt.subplots(figsize=(5,3.5)); ax.quiver([0,0],[0,0],[a[0],b[0]],[a[1],b[1]],angles='xy',scale_units='xy',scale=1,color=['navy','tomato'])
    ax.set(xlim=(-3.2,3.2),ylim=(-.3,3.2),aspect='equal',title=f'dot = {dot:.3f}; cosine = {cosine if cosine is None else round(cosine,3)}');ax.grid();plt.show();plt.close(fig)
    print('Нулевой вектор: cosine не определен.' if cosine is None else f'Квадрат расстояния после L2-нормировки = {2-2*cosine:.3f}')
    return {'dot':dot,'cosine':cosine}
panel('cosine',show_cosine,{'angle':W.IntSlider(value=45,min=0,max=180,description='Угол °',continuous_update=True),'length':W.FloatSlider(value=1,min=0,max=3,step=.25,description='Норма',continuous_update=True)})

### Вопросы и опыты · стенд 1

1. **Прогноз:** что изменится при норме 1 → 3? **Действие:** сохраните угол 45°. **Наблюдение:** выпишите dot и cosine. **Объяснение:** какая метрика реагирует на масштаб?
2. Повторите при 90° и 180°; свяжите знак cosine с геометрией.
3. Поставьте норму 0. Почему нельзя превращать неопределенное сходство в уверенное совпадение?

## 2. Калибровка порога до test
Пара принимается при $s\ge\tau$. Парный FAR делит число принятых пар **разных** ID на число таких пар; TAR — принятые пары **одного** ID на число своих пар. Для неизвестного query с галереей правило другое: ложный прием происходит, если **хотя бы один** чужой score превысил порог. Это доля запросов, не парный FAR.

Здесь 120 своих и 1280 чужих пар в каждом split, 40 неизвестных query и вложенные галереи размером 1, 4, 16, 32. Validation и test сгенерированы заранее с разными распределениями. Размер галереи меняет только набор кандидатов, а не сами scores. Это учебные числа, не ROC реальной модели.

In [ ]:
rng=np.random.default_rng(606)
VAL_GENUINE=rng.normal(.70,.10,120).clip(-1,1)
VAL_UNKNOWN=rng.normal(.42,.11,(40,32)).clip(-1,1)
TEST_GENUINE=rng.normal(.67,.11,120).clip(-1,1)
TEST_UNKNOWN=rng.normal(.45,.12,(40,32)).clip(-1,1)
SIZES=(1,4,16,32); FAR_TARGET=.05
def pair_counts(genuine,unknown,tau):
    return {'TP':int(np.count_nonzero(genuine>=tau)),
            'FP':int(np.count_nonzero(unknown>=tau)),
            'TAR':float(np.mean(genuine>=tau)),
            'FAR':float(np.mean(unknown>=tau))}
def query_false_accept(unknown,tau,size):
    return int(np.count_nonzero(unknown[:,:size].max(1)>=tau))
threshold=W.FloatSlider(value=.55,min=.45,max=.78,step=.01,description='Порог τ',continuous_update=True)
gallery_size=W.SelectionSlider(options=SIZES,value=4,description='Галерея',continuous_update=True)
forecast=W.Textarea(placeholder='До test: что ожидаете для FAR/TAR и при росте галереи?',
                    description='Прогноз',layout=W.Layout(width='95%',height='75px'))
lock=W.Button(description='Зафиксировать порог и открыть test',button_style='warning')
repeat=W.Button(description='Повторить опыт',icon='refresh')
message=W.HTML();val_out=W.Output();test_out=W.Output()
CALIBRATION={'threshold':threshold,'gallery_size':gallery_size,'forecast':forecast,
             'lock':lock,'repeat':repeat,'state':{'locked_threshold':None,'opened':False,'renders':0}}
def render_split(genuine,unknown,tau,size,label):
    pair=pair_counts(genuine,unknown,tau)
    accepts=[query_false_accept(unknown,tau,k) for k in SIZES]
    print(f'{label}: TP={pair["TP"]}/120, TAR={pair["TAR"]:.3f}; '
          f'FP={pair["FP"]}/1280, парный FAR={pair["FAR"]:.3f}')
    print(f'Неизвестные query: ложный прием {query_false_accept(unknown,tau,size)}/40 '
          f'при галерее {size}. Это доля запросов, не парный FAR.')
    fig,axes=plt.subplots(1,2,figsize=(10,3.1),constrained_layout=True)
    axes[0].hist(genuine,bins=16,alpha=.6,label='тот же ID',color='#275b8b')
    axes[0].hist(unknown.ravel(),bins=16,alpha=.55,label='другой ID',color='#cb544b')
    axes[0].axvline(tau,color='black',lw=2);axes[0].legend()
    axes[0].set(xlabel='учебный score',ylabel='число пар',title='Пары и порог')
    axes[1].plot(SIZES,np.array(accepts)/40,'o-',color='#275b8b',label='ложный прием query')
    axes[1].scatter([size],[query_false_accept(unknown,tau,size)/40],s=80,color='#cb544b')
    axes[1].axhline(pair['FAR'],ls='--',color='gray',label='парный FAR')
    axes[1].set(xlabel='число кандидатов',ylabel='доля',ylim=(-.03,1.03),
                xticks=SIZES,title='Хотя бы один score ≥ τ?')
    axes[1].legend(loc='lower right',fontsize=8)
    plt.show();plt.close(fig)
    return {'pair':pair,'query_false_accept':query_false_accept(unknown,tau,size)}
def captured_into(out,fn):
    with capture_output(display=True) as captured:result=fn()
    items=[{'output_type':'display_data','data':item.data,'metadata':item.metadata}
           for item in captured.outputs]
    if captured.stdout:items.append({'output_type':'stream','name':'stdout','text':captured.stdout})
    if captured.stderr:items.append({'output_type':'stream','name':'stderr','text':captured.stderr})
    out.outputs=tuple(items)
    return result
def refresh_calibration(change=None):
    state=CALIBRATION['state']
    tau=threshold.value if state['locked_threshold'] is None else state['locked_threshold']
    size=gallery_size.value
    state['validation']=captured_into(val_out,lambda:render_split(VAL_GENUINE,VAL_UNKNOWN,tau,size,'Validation'))
    if state['opened']:
        state['test']=captured_into(test_out,lambda:render_split(TEST_GENUINE,TEST_UNKNOWN,tau,size,'Test · порог зафиксирован'))
    else:
        test_out.outputs=({'output_type':'stream','name':'stdout',
                           'text':'Test закрыт. Запишите прогноз и зафиксируйте порог по validation.\n'},)
    state['renders']+=1
def open_test(button=None):
    if not forecast.value.strip():
        message.value='<b>Сначала запишите прогноз.</b>'
        return
    state=CALIBRATION['state'];state['locked_threshold']=float(threshold.value)
    state['opened']=True;threshold.disabled=True;forecast.disabled=True;lock.disabled=True
    far=pair_counts(VAL_GENUINE,VAL_UNKNOWN,state['locked_threshold'])['FAR']
    status='выполнено' if far<=FAR_TARGET else 'нарушено'
    message.value=(f'Порог τ={state["locked_threshold"]:.2f} зафиксирован; '
                   f'ограничение val FAR≤{FAR_TARGET:.2f}: {status}. '
                   'Размер галереи можно менять; повторный запуск не создает новый test.')
    refresh_calibration()
threshold.observe(refresh_calibration,names='value')
gallery_size.observe(refresh_calibration,names='value')
repeat.on_click(refresh_calibration);lock.on_click(open_test)
display(W.VBox([W.HBox([threshold,gallery_size]),val_out,forecast,
                W.HBox([lock,repeat]),message,test_out]))
refresh_calibration()

### Вопросы и опыты · стенд 2

1. **Прогноз:** найдите на validation порог с парным FAR≤0.05 и наибольшим доступным TAR. Запишите FP/1280, TP/120 и ожидаемое направление изменения на test. Один FP здесь меняет FAR на 1/1280; можно ли доказать FAR=$10^{-5}$?
2. **Действие:** запишите прогноз в поле и нажмите «Зафиксировать порог и открыть test». Сравните FAR/TAR на test с validation. Почему подобрать новый порог по открытому test было бы ошибкой?
3. **Прогноз и опыт:** при том же пороге увеличьте галерею 1 → 32. Запишите ложные приемы из 40 неизвестных query. Почему эта доля растет, хотя парный FAR не меняется? Объясните разницу знаменателей.

Фиксация в интерфейсе дисциплинирует один опыт; повторный запуск ячейки не создает новый независимый test.

## 3. Triplet loss: порядок недостаточен
$L=\max(0,\|a-p\|^2-\|a-n\|^2+m)$ на единичной окружности. Здесь используются **квадраты** евклидовых расстояний. Даже если positive ближе negative, margin может нарушаться.

In [ ]:
def show_triplet(positive,negative,margin):
    angles=np.deg2rad([0,positive,negative]);points=np.c_[np.cos(angles),np.sin(angles)]
    dap=float(np.sum((points[0]-points[1])**2));dan=float(np.sum((points[0]-points[2])**2));loss=max(0.,dap-dan+margin)
    fig,ax=plt.subplots(figsize=(5,3.5));t=np.linspace(0,np.pi,200);ax.plot(np.cos(t),np.sin(t),color='lightgray')
    ax.scatter(*points.T,c=['black','navy','tomato'],s=70)
    for p,s,offset in zip(points,['anchor','positive','negative'],[(-45,-15),(5,18),(5,-23)]):ax.annotate(s,p,xytext=offset,textcoords='offset points')
    ax.set(aspect='equal',xlim=(-1.3,1.4),ylim=(-.2,1.3),title=f'L = max(0, {dap:.3f} − {dan:.3f} + {margin:.2f}) = {loss:.3f}');ax.grid();plt.show();plt.close(fig)
    print('Margin нарушен: значение loss положительно. Совпавшие векторы могут дать нулевой градиент.' if loss>0 else 'Неравенство выполнено; этот triplet не дает градиента.')
    return {'loss':loss,'positive_squared':dap,'negative_squared':dan}
panel('triplet',show_triplet,{'positive':W.IntSlider(value=30,min=0,max=180,description='pos °',continuous_update=True),'negative':W.IntSlider(value=60,min=0,max=180,description='neg °',continuous_update=True),'margin':W.FloatSlider(value=.2,min=0,max=2,step=.1,description='margin',continuous_update=True)})

### Вопросы и опыты · стенд 3

1. **Прогноз:** при pos=30°, neg=60° и m=.2 loss равна нулю? Проверьте расчет.
2. Увеличьте margin до 1: что изменилось при неизменных векторах?
3. Установите neg=10°: почему это hard negative? Затем pos=neg и m=0: почему loss=0 еще не означает хорошее представление?

Статический стенд не обучает сеть и не моделирует mining батчей.

## 4. Angular margin действует при обучении
В нормированном softmax target logit равен $s\cos\theta_y$. ArcFace заменяет его на $s\cos(\theta_y+m)$, CosFace — на $s(\cos\theta_y-m)$. Другой класс остается $s\cos70^\circ$; scale=16. Margin ArcFace измеряется в **радианах**, CosFace — в единицах cosine. Численные значения margin разных методов не равноценны.

In [ ]:
def show_margin(theta,kind,margin):
    theta=np.deg2rad(theta); other=np.cos(np.deg2rad(70))
    base=np.cos(theta)
    target={'none':base,'ArcFace':np.cos(theta+margin),'CosFace':base-margin}[kind]
    logits=16*np.array([target,other]); p=np.exp(logits-logits.max());p/=p.sum()
    fig,ax=plt.subplots(figsize=(5,2.8));ax.bar(['target logit','other logit'],logits,color=['navy','tomato']);ax.set(ylabel='logit',title=f'{kind}: p(target)={p[0]:.4f}; CE={-np.log(p[0]):.3f}');plt.show();plt.close(fig)
    print('Margin применяется к известному target только при обучении; поисковый cosine при инференсе не меняется.')
    return {'target':float(target),'p_target':float(p[0])}
panel('angular',show_margin,{'theta':W.IntSlider(value=50,min=0,max=120,description='theta °',continuous_update=True),'kind':W.Dropdown(options=['none','ArcFace','CosFace'],value='ArcFace',description='Loss'),'margin':W.FloatSlider(value=.35,min=0,max=.6,step=.05,description='margin',continuous_update=True)})

### Вопросы и опыты · стенд 4

1. **Прогноз:** при theta=50° переход none → ArcFace с m=.35 понизит вероятность target? Проверьте вероятность и CE.
2. При m=0 сопоставьте все варианты. Почему они совпадают?
3. При theta=70° и m>0 объясните, почему loss требует большего сближения target.

Углы ограничены так, чтобы theta+m<π; производственные варианты учитывают монотонность и устойчивость. SphereFace использует мультипликативный угловой margin; здесь его piecewise-формула не имитируется упрощенной cosine.

## 5. Разбор нескольких query/gallery
Пять query A–E относятся к камере 0; в общей галерее 16 наблюдений. Сходства **заданы учебными числами**; ползунок меняет score одного чужого ID X у всех запросов при фиксированных остальных scores. В выбранном протоколе, по принципу оценки Market-1501, исключается **same ID AND same camera**. AP — средняя precision на позициях релевантных наблюдений; rank-1 проверяет только первое. Query без допустимых positives пропускается из среднего и считается отдельно.

In [ ]:
Q_IDS=np.array(list('ABCDE')); Q_CAMS=np.zeros(5,dtype=int)
G_IDS=np.array(list('AAABBBCDDDEEE'+'XYZ'))
G_CAMS=np.array([0,1,2, 0,1,2, 0, 0,1,2, 0,1,2, 1,0,2])
# Ряды A–E, столбцы галереи. Это придуманные scores, не выход модели.
BASE_SCORES=np.array([
 [.99,.92,.70,.45,.76,.52,.50,.35,.60,.48,.55,.58,.41,0,.66,.31],
 [.40,.45,.42,.99,.86,.83,.20,.30,.40,.20,.25,.30,.20,0,.50,.40],
 [.45,.31,.25,.46,.35,.25,.99,.48,.28,.39,.42,.37,.30,0,.55,.40],
 [.48,.62,.20,.25,.57,.33,.20,.99,.89,.55,.20,.28,.35,0,.50,.31],
 [.45,.60,.52,.55,.71,.42,.31,.30,.62,.40,.99,.78,.74,0,.69,.30]
])
def show_gallery(query,hard_negative,protocol):
    scores=BASE_SCORES.copy()
    scores[:,13]=hard_negative+np.array([0,-.06,-.10,.03,-.14])
    cases=[]
    for qi,qid in enumerate(Q_IDS):
        if protocol=='correct':exclude=(G_IDS==qid)&(G_CAMS==Q_CAMS[qi])
        elif protocol=='camera':exclude=G_CAMS==Q_CAMS[qi]
        else:exclude=np.zeros(len(G_IDS),dtype=bool)
        allowed=np.flatnonzero(~exclude)
        order=allowed[np.argsort(-scores[qi,allowed],kind='stable')]
        relevant=G_IDS[order]==qid
        positions=np.flatnonzero(relevant)+1
        terms=np.arange(1,len(positions)+1)/positions if len(positions) else np.array([])
        cases.append({'order':order,'positions':positions,'terms':terms,
                      'AP':float(terms.mean()) if len(terms) else None,
                      'rank1':int(relevant[0]) if len(terms) else None,
                      'excluded':np.flatnonzero(exclude).tolist()})
    valid=[c for c in cases if c['AP'] is not None]
    rank1=float(np.mean([c['rank1'] for c in valid])) if valid else float('nan')
    mAP=float(np.mean([c['AP'] for c in valid])) if valid else float('nan')
    qi=int(np.flatnonzero(Q_IDS==query)[0]);case=cases[qi];order=case['order'][:8]
    fig,axes=plt.subplots(1,2,figsize=(10,3.8),constrained_layout=True)
    labels=[f'{G_IDS[j]}/{G_CAMS[j]} · #{j}' for j in order]
    colors=['#275b8b' if G_IDS[j]==query else '#cb544b' for j in order]
    axes[0].barh(labels,scores[qi,order],color=colors)
    axes[0].invert_yaxis();axes[0].set(xlim=(0,1),xlabel='учебный score',
                                   title=f'Query {query}/0 · первые 8 допустимых')
    axes[1].bar(Q_IDS,[c['AP'] if c['AP'] is not None else 0 for c in cases],
                color=['#aab2bb' if c['AP'] is None else ('#275b8b' if c['rank1'] else '#cb544b') for c in cases])
    axes[1].set(ylim=(0,1.08),ylabel='AP запроса',title=f'rank-1={rank1:.3f}; mAP={mAP:.3f}')
    for j,c in enumerate(cases):
        if c['AP'] is None:axes[1].text(j,.04,'SKIP',ha='center',va='bottom',fontsize=9,color='#53606d')
    axes[1].grid(axis='y',alpha=.2)
    plt.show();plt.close(fig)
    if case['AP'] is None:
        print(f'Query {query}: SKIP — после исключений нет допустимого positive.')
    else:
        parts=' + '.join(f'{v:.3f}@{p}' for p,v in zip(case['positions'],case['terms']))
        print(f'Query {query}: AP=({parts})/{len(case["terms"])}={case["AP"]:.3f}; rank-1={case["rank1"]}')
    print(f'Исключены индексы {case["excluded"]}; valid={len(valid)}, skipped={len(cases)-len(valid)}.')
    print('Синие столбцы — свой ID, красные — чужой; серый AP=0 на рисунке означает SKIP, а не нулевой AP.')
    return {'rank1':rank1,'mAP':mAP,'valid_queries':len(valid),
            'skipped_queries':len(cases)-len(valid),'per_query':cases,
            'selected_AP':case['AP'],'selected_rank1':case['rank1']}
panel('gallery',show_gallery,{
    'query':W.Dropdown(options=list(Q_IDS),value='A',description='Query'),
    'hard_negative':W.FloatSlider(value=.50,min=.50,max=.96,step=.02,description='Двойник X',continuous_update=True),
    'protocol':W.Dropdown(options=[('ID и камера','correct'),('Без исключений','none'),('Вся камера','camera')],value='correct',description='Исключения')})

### Вопросы и опыты · стенд 5

1. **Прогноз:** для query A увеличьте score двойника X с .50 до .90. Сравните позиции positives, слагаемые AP, rank-1 запроса и общие rank-1/mAP. Найдите момент, когда AP уже ухудшилась, а rank-1 запроса еще прежний.
2. Перейдите к C. Почему этот query пропущен при правильном исключении? Запишите число valid/skipped; серый столбец — отметка пропуска, не AP=0.
3. Смените правило на «без исключений», затем на «вся камера». На примере A и C объясните оба нарушения. Чужой ID Y камеры 0 при правильном протоколе остается допустимым negative.

Не сравнивайте значения mAP между разными протоколами как улучшение признаков. Tie разрешается стабильным исходным порядком. В реальном наборе также нужны официальные правила junk и distractors.

## 6. Камера как ложный признак
Для восьми новых validation ID заданы одни и те же 48 синтетических признаков: identity + сдвиг камеры + шум. Query — первое наблюдение камеры 0, кандидаты — все записи; same-ID/same-camera исключаются. Сдвиги камеры заранее оценены на **отдельных train ID**. При сбалансированном train каждый ID был во всех трех камерах; при смешении camera и identity разные камеры видели разные группы ID. Test в этом стенде нет.

In [ ]:
# Малый независимый validation-набор; offsets рассчитаны в закрытом train с ID 0–11.
# Здесь только новые ID 100–107. Это учебные признаки, не эмбеддинги модели.
EVAL_X=np.array([[0.4944376957217465, 0.18528799504023996, 0.7240267388174858, 1.2515457325182613], [0.49436516779670936, 0.2319824306076295, 0.781024986418553, 1.2725126291295576], [-0.008999304861889178, 0.3681501826791357, 0.7585253894059866, 1.2935739434269544], [-0.04468262053302345, 0.5027177004887495, 0.7827325242235113, 1.2310491656737148], [-0.002047413230154338, -0.18292584886207097, 0.7519686538440779, 1.2718743815766942], [-0.056209540810628, -0.08785494371328176, 0.7597502268069746, 1.3421915214825875], [1.2633695307583896, 0.25798109616807924, -0.33735877209528137, -1.1158878425184828], [1.239109084827549, 0.26018019899626904, -0.38536615245762745, -1.150956744629869], [0.6355512750576033, 0.5436794381042435, -0.45596584593932327, -1.1485389395502694], [0.6242838463718132, 0.5436841525723173, -0.35709189581568057, -1.137029317079857], [0.7700870038640433, -0.024504044828361382, -0.4251201264929628, -1.1461837482820607], [0.6458421212478684, -0.014754589054063787, -0.3887332677516503, -1.1881414123593435], [-0.6018341507490276, -0.6970211240373906, 0.7685563783339, -0.4913364344678993], [-0.5769715169620729, -0.6056185526525473, 0.8451365270308616, -0.47650996716253546], [-1.0722790415602366, -0.3355728644975087, 0.939771691901921, -0.42866727189557674], [-1.102243476544939, -0.29562654078337913, 0.9362695242114498, -0.4816917699523736], [-1.0933639682094898, -1.0527750206657607, 0.841589054309602, -0.5021104274033121], [-1.1119745855275065, -0.9928247176803028, 0.8514929346986085, -0.46307457596741697], [-0.6724029825719923, 0.5394490809033909, -0.7050914184901658, -0.7099190111855098], [-0.6353338000800377, 0.5987672778701956, -0.699723451652283, -0.6173901438655854], [-1.2983816337976768, 0.7528914088086958, -0.6904954284990195, -0.5718767747377475], [-1.2744039630368862, 0.7892959422061829, -0.5966214647868113, -0.7401332554400043], [-1.1897652760226085, 0.24919285271242875, -0.6873147413085042, -0.6360296112901407], [-1.0909893060166764, 0.1684599968204207, -0.6926982122890449, -0.6573670304110857], [0.19037659525886927, 0.19459675049215752, -1.2549455458017362, -0.6092151057767754], [0.13696104312153456, 0.22655068256070143, -1.3143405243066413, -0.60004885301974], [-0.3721557060343466, 0.5602195513464223, -1.3215664724579264, -0.6005912669460124], [-0.2804093702697803, 0.4736401836502543, -1.2422614185639549, -0.6096631790433803], [-0.20983986388748993, 0.10174584415372298, -1.3426384276646046, -0.6819079944244074], [-0.25693282330729544, -0.050450783184931354, -1.3571810494732395, -0.6718841789400378], [0.6744195409862991, -0.5199416962407879, -1.2518113230736982, -0.7625444938880366], [0.5321673705245336, -0.4899127649126453, -1.2028003127072653, -0.793037135372195], [0.09511862052932575, -0.22687966882254137, -1.1456440223492865, -0.677116749344996], [0.046932673829837804, -0.22703369182706318, -1.1718672808060417, -0.6955934474743806], [0.06489065950994789, -0.7939101806930401, -1.1831766237647294, -0.7246051934641233], [0.09646746322367167, -0.8362111193946777, -1.2059587759088821, -0.7379864781535678], [1.4672691290753188, -0.37170970349849763, 0.6187283257284998, 0.6566196453864934], [1.4703782523504096, -0.41737609224139605, 0.6794821418328616, 0.7016310303086086], [0.919559661101336, -0.11746731460204747, 0.6856131054697705, 0.7332706235589895], [0.9819645518835088, -0.017523531010957834, 0.5175718050564704, 0.6577713360387111], [1.0125961950518494, -0.7746143435710556, 0.5527711584530772, 0.6968484806994435], [1.0321372301067602, -0.7112813498292332, 0.6064113419048498, 0.737741365992888], [-0.3531505401661453, 0.6070813695079249, -0.687149487304164, -0.9471100752365855], [-0.2690014720601169, 0.681755350175704, -0.7241838365227847, -0.8074249646942182], [-0.8840597359688506, 1.0004718620802575, -0.6502675298397086, -0.8882360888029359], [-0.8504080909748918, 1.0061832641194992, -0.6874742134652844, -0.9984112919490945], [-0.8700444168297927, 0.36072896914860414, -0.6929651198488829, -0.9206476064736705], [-0.8902890300963193, 0.43681244656109347, -0.6057433619926839, -1.037470273978013]])
EVAL_IDS=np.repeat(np.arange(100,108),6)
EVAL_CAMS=np.tile(np.repeat(np.arange(3),2),8)
OFFSETS={'balanced':np.array([[0.3315930940274766, 0.0031199659871042618, -0.002134935140656574, 0.01111544807517717], [-0.1638209798025176, 0.2957116487905912, 0.0009006235526917983, 0.0020420783378729546], [-0.16777211422495925, -0.2988316147776955, 0.0012343115879648037, -0.013157526413050291]]),
         'confounded':np.array([[-0.0968047517453334, 0.09543919248538585, -0.013890545113196597, -0.00418079925788678], [0.09356282674223995, -0.10958336110262457, -0.01117595639220188, 0.03892817454168862], [0.0032419250030934557, 0.014144168617238675, 0.025066501505398475, -0.034747375283801826]])}
def camera_case(scenario,alpha):
    x=EVAL_X-alpha*OFFSETS[scenario][EVAL_CAMS]
    x=x/np.linalg.norm(x,axis=1,keepdims=True)
    cases=[]
    for qi in range(8):
        qindex=qi*6; qid=EVAL_IDS[qindex]
        keep=~((EVAL_IDS==qid)&(EVAL_CAMS==0))
        available=np.flatnonzero(keep)
        scores=x[available]@x[qindex]
        order=available[np.argsort(-scores,kind='stable')]
        rel=EVAL_IDS[order]==qid
        positions=np.flatnonzero(rel)+1
        precision=np.arange(1,len(positions)+1)/positions
        cases.append({'AP':float(precision.mean()),'rank1':int(rel[0]),
                      'top_id':int(EVAL_IDS[order[0]]),'order':order,
                      'scores':scores[np.argsort(-scores,kind='stable')],
                      'positions':positions,'precision':precision})
    return {'mAP':float(np.mean([c['AP'] for c in cases])),
            'rank1':float(np.mean([c['rank1'] for c in cases])),'cases':cases}
def show_camera(scenario,alpha,query):
    baseline=camera_case(scenario,0.);current=camera_case(scenario,alpha)
    i=query-100;old=baseline['cases'][i];new=current['cases'][i]
    order=new['order'][:10]
    fig,axes=plt.subplots(1,2,figsize=(11,4),constrained_layout=True)
    labels=[f'{EVAL_IDS[j]}/{EVAL_CAMS[j]} · #{j}' for j in order]
    axes[0].barh(labels,new['scores'][:10],
                 color=['#275b8b' if EVAL_IDS[j]==query else '#cb544b' for j in order])
    axes[0].invert_yaxis();axes[0].set(xlim=(-1,1),xlabel='cosine',
        title=f'Query {query}/0 · первые 10 кандидатов')
    positions=np.arange(8)
    axes[1].bar(positions-.18,[c['AP'] for c in baseline['cases']],.36,
                color='#aab2bb',label='alpha=0')
    axes[1].bar(positions+.18,[c['AP'] for c in current['cases']],.36,
                color='#275b8b',label=f'alpha={alpha:g}')
    axes[1].set(xticks=positions,xticklabels=np.arange(100,108),ylim=(0,1.05),
                xlabel='тот же query ID',ylabel='AP',title='До и после на тех же запросах')
    axes[1].legend(loc='lower left');plt.show();plt.close(fig)
    print(f'{scenario}, alpha={alpha:g}: rank-1 {baseline["rank1"]:.3f} → {current["rank1"]:.3f}; '
          f'mAP {baseline["mAP"]:.3f} → {current["mAP"]:.3f}')
    print(f'Query {query}: top ID {old["top_id"]} → {new["top_id"]}; '
          f'AP {old["AP"]:.3f} → {new["AP"]:.3f}; '
          f'позиции positives {new["positions"].tolist()}')
    return {'baseline_rank1':baseline['rank1'],'baseline_mAP':baseline['mAP'],
            'rank1':current['rank1'],'mAP':current['mAP'],
            'selected_AP':new['AP'],'selected_top_id':new['top_id']}
panel('camera',show_camera,{
    'scenario':W.Dropdown(options=[('Сбалансированный train','balanced'),
                                   ('ID зависит от камеры','confounded')],value='balanced',description='Train'),
    'alpha':W.SelectionSlider(options=[0.,.25,.5,.75,1.],value=0.,description='alpha',continuous_update=True),
    'query':W.Dropdown(options=list(range(100,108)),value=103,description='Query ID')})

### Вопросы и опыты · стенд 6

1. **Прогноз:** при сбалансированном train что даст alpha 0 → 1? Зафиксируйте query 103, сравните тот же первый найденный ID, AP, общий rank-1/mAP и диаграмму AP всех восьми запросов.
2. Сохраните alpha=1 и переключите train на «ID зависит от камеры». Что произошло с теми же validation запросами? Найдите query, у которого коррекция испортила первое место; объясните, почему среднее камеры могло содержать identity.
3. Верните alpha=0 и проверьте, что оба сценария дают одинаковый baseline. Измените только alpha по сетке .25/.5/.75/1 и сделайте вывод о причинном факторе. Почему успех на этих данных не доказывает работу на неизвестной камере?

Векторные признаки и заранее оцененные сдвиги созданы отдельно для демо; это не результаты FaceNet/OSNet и не готовое решение самостоятельного TODO.

## Мост к самостоятельной работе
В 02 вы самостоятельно реализуете полный оценщик, корректное разделение по ID, калибровку порога по val и оценку camera offsets **только на train**. Данные и численные ответы 02 отличаются от этих учебных опытов. Центральный вопрос — что именно измеряет протокол и где он перестает работать.

Источники: [FaceNet, CVPR 2015](https://arxiv.org/abs/1503.03832), [ArcFace, CVPR 2019](https://arxiv.org/abs/1801.07698), [официальный Market-1501](https://zheng-lab-anu.github.io/Project/project_reid.html). Данные и рисунки этой работы генерируются локально; чужие фотографии и веса не включены.